# Steel decarbonisation study: re-run and cross-check in Google Colab

This notebook re-runs the key steps of the study by hand and checks the repository's numbers by an
independent route (own formulas, not the functions under test). Each check prints PASS or FAIL with its tolerance.

**Honest scope**
- The plant is hypothetical (120,000 t/yr TMT rebar, SIPCOT Ranipet).
- Prices, tariffs, emission factors and node locations are real and sourced in `data/inputs.csv`.
- The quantum kernel is classically simulated (state-vector). No hardware, no speed-up claim.
- The full run (15 runs x budget 10000) is for a many-core PC, not free Colab. Here you run the quick smoke run
  and then check the author's full `results_v2/` files.

Run the cells in order.

In [ ]:
# Setup: get the repo into Colab, pin versions, pin BLAS to 1 thread. PASS = the last line prints the repo root.
import os, sys, glob, subprocess
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"          # must be set before numpy is imported

# Option A: upload steel_decarb.zip (zip of the repo folder contents).
from google.colab import files
up = files.upload()              # choose steel_decarb.zip
# Option B: mount Google Drive instead (comment out Option A, uncomment these lines).
# from google.colab import drive
# drive.mount('/content/drive')
# subprocess.run(['cp', '/content/drive/MyDrive/steel_decarb.zip', '.'], check=True)

!rm -rf steel_decarb && mkdir steel_decarb && unzip -q -o steel_decarb.zip -d steel_decarb
%pip install -q numpy==2.4.4 scipy==1.17.1 matplotlib
runall = glob.glob('steel_decarb/**/run_all.py', recursive=True)
assert runall, 'run_all.py not found in the zip'
ROOT = os.path.dirname(os.path.abspath(runall[0]))
os.chdir(ROOT)
sys.path.insert(0, 'src')
sys.path.insert(0, 'colab')
print('repo root:', ROOT)

In [ ]:
# Data check: every id loads, triangular rows have low <= mode <= high, required ids are present.
# PASS = status PASS and loaded count equals the CSV id count.
import crosscheck as cc
r = cc.check_inputs('.')
cc.print_inputs_table(r['table'])
print('problems:', r['problems'])
print('inputs', 'PASS' if r['ok'] else 'FAIL', 'loaded', r['value_ours'], 'csv ids', r['value_check'])

In [ ]:
# Base stochastic MILP step by step, then the objective recomputed by hand from the solution vector.
# PASS = hand objective equals solver objective within 1e-9 relative and evaluate() reports no violation.
from steel.data import load_inputs
from steel.model import build, solve
from steel.encoding import evaluate, TOL
inp = load_inputs('data/inputs.csv', 'data/structure.json')
scens = cc.scenarios_for('.', 'results/quick')       # price scenarios (Monte Carlo, k-means reduction)
prob = build(inp, scens)
sol = solve(prob)
chosen = [n for n in prob.y_names if sol.z[prob.iy(n)] > 0.5]
print('scenarios:', len(scens), ' chosen investments (y = 1):', chosen)
print('expected cost (INR):', f'{sol.cost:,.0f}', ' expected emissions (tCO2):', f'{sol.emis:,.1f}')
hand = cc.hand_objective(inp, scens, prob, sol.z)
cost_e, emis_e, viol = evaluate(prob, sol.z)
print('hand objective  :', f'{hand:,.2f}')
print('relative diff   :', abs(hand - sol.cost) / sol.cost, ' evaluate() violation:', viol, '(limit', TOL, ')')
r = cc.check_objective('.', scens)
print('objective', 'PASS' if r['ok'] else 'FAIL', '(tol', r['tol'], ')')

In [ ]:
# Pareto front (cost vs emissions) with the epsilon-constraint MILP, plus its checks.
# PASS = min-cost end equals the base MILP cost, costs do not fall as emissions tighten, file matches.
import matplotlib.pyplot as plt
from steel.model import pareto
front = pareto(inp, scens, n=5)
eps = [e for e, _, _ in front]; cost = [c for _, c, _ in front]
plt.plot(eps, [c / 1e7 for c in cost], 'o-'); plt.xlabel('emission cap (tCO2)'); plt.ylabel('expected cost (crore INR)')
plt.title('Pareto front'); plt.show()
r = cc.check_pareto('.', 'results/quick', scens)
print('pareto', 'PASS' if r['ok'] else 'FAIL', 'monotone:', r['monotone'], 'file match:', r['file_match'], '(tol', r['tol'], ')')

In [ ]:
# Moran's I by hand: I = (n/S0) (z'Wz)/(z'z), z centred, own kNN weights, plus a 999-permutation p.
# PASS = Moran's I equals results/quick/moran.csv within 1e-9 for all four weight sets.
import numpy as np
nodes = cc._rows('data/nodes.csv')
z = np.array([float(n['ef_prod']) + float(n['ef_transport']) * float(n['dist_km']) for n in nodes])
xy = cc._project([float(n['lat']) for n in nodes], [float(n['lon']) for n in nodes])
W = cc._knn_row_standardised(xy, 4)
zc = z - z.mean()
print('n =', len(z), ' S0 =', W.sum(), ' Moran I (knn4) =', cc._moran(z, W))
r = cc.check_moran('.', 'results/quick')
print('permutation p (999 perms):', r['perm_p'])
plt.scatter(zc, W @ zc); plt.xlabel('centred intensity'); plt.ylabel('mean of 4 nearest neighbours')
plt.title("Moran scatter (knn4)"); plt.show()
print('moran', 'PASS' if r['ok'] else 'FAIL', '(tol', r['tol'], ')')

In [ ]:
# Quantum kernel by hand: build the ZZ feature-map state gate by gate, then |<psi(x)|psi(y)>|^2.
# PASS = matches steel.qkernel.zz_kernel within 1e-10; matrix symmetric, diagonal 1, smallest eigenvalue >= -1e-10.
r = cc.check_kernel()
print('max |hand - library| =', r['max_abs_diff'], ' asymmetry =', r['asym'], ' diag error =', r['diag_err'], ' min eig =', r['min_eig'])
print('kernel', 'PASS' if r['ok'] else 'FAIL', '(tol', r['tol'], ')')

In [ ]:
# Optional: regenerate the quick results here (about 2-5 min on Colab), then cross-check them.
# PASS = every row of the summary table says PASS.
!python run_all.py --quick
res = cc.run_all('.', 'results/quick')
print(cc.summary_table(res))

In [ ]:
# Cross-check the author's full results: upload results.zip (zip of the results_v2/ folder), unzip, run all checks.
# PASS = every row says PASS. Full results use 5000 samples and 9999 permutations; the checks read those from the files.
from google.colab import files
up = files.upload()              # choose results.zip
!rm -rf results_author && mkdir results_author && unzip -q -o results.zip -d results_author
found = glob.glob('results_author/**/bench.csv', recursive=True)
assert found, 'bench.csv not found in results.zip'
RES = os.path.dirname(found[0])
res = cc.run_all('.', RES)
print(cc.summary_table(res))
print('ALL PASS' if all(x['ok'] for x in res) else 'SOME CHECKS FAILED')